# Tutorial: Causal Machine Learning and the Resource Curse (EconML)

**Double Machine Learning Causal Forest with Simulated Data**

Based on Hodler, Lechner & Raschky (2023) "Institutions and the resource curse: New insights from causal machine learning" (*PLoS ONE*).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/starter-academic-v501/blob/master/content/post/python_EconML/notebook.ipynb)

---

## Abstract

This notebook teaches the **EconML Causal Forest** (`CausalForestDML`) estimator on a simulated panel where the ground-truth treatment effects are known. The simulated DGP mirrors Hodler, Lechner & Raschky (2023) and embeds three findings to recover: (1) mining raises economic activity, (2) the price gradient is non-linear, and (3) institutions moderate the mining effect but not the price effect. Methodologically, the notebook focuses on the Double Machine Learning argument that lets a causal forest deliver $\sqrt{n}$-consistent CATE estimates even with flexible first-stage learners, and on the practical machinery of `CausalForestDML`: cross-fitting, honest splitting, BLB inference, GATEs, feature importance, and the `SingleTreeCateInterpreter`.

The notebook is the runnable companion to the blog post [Causal Machine Learning and the Resource Curse with Python EconML](https://carlos-mendez.org/post/python_econml/).

## 1. Setup and Installation

In [ ]:
# Install dependencies (Colab only)
!pip install -q econml matplotlib pandas numpy

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import sklearn
import econml

# Environment fingerprint — pin numbers to a specific stack so future
# audits can detect drift caused by version updates.
print('Environment:')
print(f'  Python:       {sys.version.split()[0]}')
print(f'  econml:       {econml.__version__}')
print(f'  scikit-learn: {sklearn.__version__}')
print(f'  numpy:        {np.__version__}')
print(f'  pandas:       {pd.__version__}')
print(f'  matplotlib:   {matplotlib.__version__}')

# Site color palette — dark theme
STEEL_BLUE = '#6a9bcc'
WARM_ORANGE = '#d97757'
NEAR_BLACK = '#141413'
TEAL = '#00d4c8'
DARK_NAVY = '#0f1729'
GRID_LINE = '#1f2b5e'
LIGHT_TEXT = '#c8d0e0'
WHITE_TEXT = '#e8ecf2'

plt.rcParams.update({
    'figure.facecolor': DARK_NAVY,
    'axes.facecolor': DARK_NAVY,
    'axes.edgecolor': DARK_NAVY,
    'axes.linewidth': 0,
    'axes.labelcolor': LIGHT_TEXT,
    'axes.titlecolor': WHITE_TEXT,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': False,
    'axes.spines.bottom': False,
    'axes.grid': True,
    'grid.color': GRID_LINE,
    'grid.linewidth': 0.6,
    'grid.alpha': 0.8,
    'xtick.color': LIGHT_TEXT,
    'ytick.color': LIGHT_TEXT,
    'xtick.major.size': 0,
    'ytick.major.size': 0,
    'text.color': WHITE_TEXT,
    'font.size': 12,
    'legend.frameon': False,
    'legend.fontsize': 11,
    'legend.labelcolor': LIGHT_TEXT,
    'figure.edgecolor': DARK_NAVY,
    'savefig.facecolor': DARK_NAVY,
    'savefig.edgecolor': DARK_NAVY,
})

### Ground-Truth Parameters

The simulated data embeds three findings with known causal parameters:

In [ ]:
# Ground-truth parameters embedded in the DGP
TRUE_PARAMS = {
    'ntl_mining_base': 0.25,         # Mining effect at mean institutions
    'ntl_mining_inst_mod': 0.15,     # Institutional moderation of mining
    'ntl_price_med': 0.05,           # Medium price premium (small)
    'ntl_price_high': 0.30,          # High price premium (large)
    'ntl_noise_sd': 0.25,            # Outcome noise
    'conflict_mining_base': 0.70,    # Mining increases conflict
    'conflict_mining_inst_mod': -0.50,  # Institutions dampen mining-conflict
    'conflict_price_med': 0.15,
    'conflict_price_high': 0.50,
    'conflict_base_rate': 0.12,
}

def expected_ates_ntl():
    p = TRUE_PARAMS
    return {
        '1-0': p['ntl_mining_base'],
        '2-0': p['ntl_mining_base'] + p['ntl_price_med'],
        '3-0': p['ntl_mining_base'] + p['ntl_price_high'],
        '2-1': p['ntl_price_med'],
        '3-1': p['ntl_price_high'],
        '3-2': p['ntl_price_high'] - p['ntl_price_med'],
    }

print("Ground-truth NTL ATEs:")
for comp, val in expected_ates_ntl().items():
    print(f"  ATE({comp}) = {val:.3f}")

## 2. Load Simulated Data

The simulated dataset (3,000 observations, 300 districts, 8 countries) uses known ground-truth causal effects embedded in the data-generating process.

In [ ]:
# Load from GitHub (works in Colab); fall back to a local CSV if the
# URL is unreachable.
DATA_URL = ('https://github.com/cmg777/starter-academic-v501'
            '/raw/master/content/post/python_EconML/sim_resource_curse.csv')

try:
    df = pd.read_csv(DATA_URL)
except Exception:
    df = pd.read_csv('sim_resource_curse.csv')

print(f'Loaded {len(df):,} observations')
print(f"Districts: {df['district_id'].nunique()}, "
      f"Countries: {df['country_id'].nunique()}, "
      f"Years: {df['year'].min()}-{df['year'].max()}")
print(f"Mining districts: {df.loc[df['mining']==1, 'district_id'].nunique()} "
      f"({df['mining'].mean():.0%})")

## 3. Descriptive Statistics

In [ ]:
labels = {0: 'No mining', 1: 'Low prices', 2: 'Med prices', 3: 'High prices'}

# Treatment distribution chart
fig, ax = plt.subplots(figsize=(8, 3))
fig.patch.set_linewidth(0)
colors = [LIGHT_TEXT, STEEL_BLUE, TEAL, WARM_ORANGE]
counts = df['treatment'].value_counts().sort_index()
bars = ax.barh([labels[t] for t in counts.index], counts.values,
               color=colors, edgecolor=DARK_NAVY, linewidth=0.8)
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{count:,} ({count/len(df):.1%})', va='center',
            fontsize=9, color=LIGHT_TEXT)
ax.set_xlabel('Number of observations')
ax.set_title('Treatment Distribution (M=4)')
plt.tight_layout()
plt.show()

In [ ]:
# Outcomes by treatment group
print(f"{'Treatment':<20s} {'Mean NTL':>10s} {'Conflict Rate':>15s}")
print("-" * 47)
for t in sorted(df['treatment'].unique()):
    mask = df['treatment'] == t
    m_ntl = df.loc[mask, 'ntl_log'].mean()
    m_conf = df.loc[mask, 'conflict'].mean()
    print(f"{t} ({labels[t]:<13s})  {m_ntl:>10.3f} {m_conf:>14.1%}")

### Naive Comparison: Why We Need Causal ML

Simple difference-in-means is **biased** because mining districts differ systematically from non-mining districts. The DML Causal Forest corrects for this by residualizing both outcomes and treatments.

In [ ]:
gt = expected_ates_ntl()

print("Simple difference-in-means (no confounder adjustment):")
print(f"{'Comparison':<15s} {'Naive':>8s} {'Ground Truth':>14s} {'Bias':>8s}")
print("-" * 47)

for comp in ['1-0', '2-1', '3-1']:
    a, b = int(comp[0]), int(comp[2])
    mean_a = df.loc[df['treatment'] == a, 'ntl_log'].mean()
    mean_b = df.loc[df['treatment'] == b, 'ntl_log'].mean()
    naive = mean_a - mean_b
    truth = gt[comp]
    bias = naive - truth
    print(f"{comp:<15s} {naive:>8.3f} {truth:>14.3f} {bias:>+8.3f}")

## 4. EconML Causal Forest Estimation

### Potential outcomes and the CATE

Causal inference rests on the **potential-outcomes** framework. For each unit $i$ and treatment value $t$, $Y_i(t)$ is the outcome that *would* be realized if $i$ received treatment $t$. The fundamental problem of causal inference is that we observe only one of these per unit; the rest are counterfactual. The **Conditional Average Treatment Effect** is

$$\tau(\mathbf{x}) = E\{Y_i(1) - Y_i(0) \mid \mathbf{X}_i = \mathbf{x}\}.$$

When $\tau(\mathbf{x})$ bends with $\mathbf{x}$, we have **treatment effect heterogeneity** — mining might raise nighttime lights in well-governed districts and barely move them elsewhere. Estimating that bend is the point of a causal forest.

### The partially linear model with heterogeneous effects

EconML's `CausalForestDML` works inside the partially linear model of Robinson (1988), extended by Chernozhukov et al. (2018) to allow heterogeneous effects:

$$Y_i = \tau(\mathbf{X}_i)\, T_i + g_0(\mathbf{X}_i, \mathbf{W}_i) + \varepsilon_i, \qquad E[\varepsilon_i \mid \mathbf{X}_i, \mathbf{W}_i] = 0.$$

$$T_i = m_0(\mathbf{X}_i, \mathbf{W}_i) + v_i, \qquad E[v_i \mid \mathbf{X}_i, \mathbf{W}_i] = 0.$$

$g_0$ and $m_0$ are **nuisance functions**. We do not care about their values — we estimate them only to remove confounding from the second stage. For our four-level treatment, $m_0$ is a multi-class classifier (specifically `GradientBoostingClassifier`).

### Why two stages? The residualization argument

Subtracting $E[Y_i \mid \mathbf{X}, \mathbf{W}]$ from the outcome equation cancels the $g_0$ term and yields

$$\underbrace{Y_i - E[Y_i \mid \mathbf{X}, \mathbf{W}]}_{\tilde Y_i} = \tau(\mathbf{X}_i) \cdot \underbrace{(T_i - m_0(\mathbf{X}_i, \mathbf{W}_i))}_{\tilde T_i} + \varepsilon_i.$$

So if we (a) estimate $g_0$ and $m_0$ in a first stage with any flexible learner, (b) residualize both $Y$ and $T$, and (c) regress $\tilde Y$ on $\tilde T$ with covariate-dependent slope, that slope at point $\mathbf{x}$ recovers $\tau(\mathbf{x})$. This is the Frisch–Waugh–Lovell logic.

### Neyman orthogonality

The DML estimating equation $\psi(W; \tau, \eta)$ — where $\eta = (g_0, m_0)$ — satisfies

$$\left.\frac{\partial}{\partial \eta} E[\psi(W; \tau, \eta)] \right|_{\eta = \eta_0} = 0.$$

At the truth, the expected estimating equation is *flat* in the nuisance functions. Even if Gradient Boosting estimates $g_0, m_0$ at the slow rate $O(n^{-1/4})$, the resulting estimate of $\tau$ is still $\sqrt{n}$-consistent (Chernozhukov et al., 2018, §2.2).

### X (heterogeneity features) vs W (controls)

- **X**: variables the causal forest can split on to discover heterogeneity (e.g., institutions, geography).
- **W**: variables used only in the first-stage nuisance models (here, country and year fixed effects). These soak up confounding without entering the heterogeneity surface.

In [ ]:
from econml.dml import CausalForestDML
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier

# Heterogeneity features (X): causal forest splits on these
X_COLS = [
    'exec_constraints', 'quality_of_govt', 'gdp_pc',
    'elevation', 'temperature', 'ruggedness',
    'distance_capital', 'agri_suitability', 'population', 'ethnic_frac',
]

# Additional controls (W): first-stage only
W_COLS = ['country_id', 'year']


def fit_causal_forest(df, outcome, label):
    """Fit CausalForestDML for one outcome.

    Mirrors script.py:step_econml_estimation. Update both together.
    """
    Y = df[outcome].values
    T = df['treatment'].values
    X = df[X_COLS].values
    W = df[W_COLS].values

    est = CausalForestDML(
        model_y=GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                          random_state=42),
        model_t=GradientBoostingClassifier(n_estimators=200, max_depth=4,
                                           random_state=42),
        discrete_treatment=True,
        categories=[0, 1, 2, 3],
        n_estimators=500,
        min_samples_leaf=10,
        max_depth=None,
        honest=True,        # honesty enables valid CIs
        inference=True,     # BLB (bootstrap of little bags)
        cv=5,               # 5-fold cross-fitting
        n_jobs=1,
        random_state=42,
    )
    t0 = time.time()
    print(f'Fitting {label} (first-stage residualization + causal forest)...')
    # groups=district_id triggers GroupKFold for cross-fitting only.
    # It prevents within-district leakage across folds; it does NOT
    # translate into clustered second-stage SEs.
    est.fit(Y, T, X=X, W=W, groups=df['district_id'].values)
    print(f'Done in {(time.time() - t0)/60:.1f} minutes')
    return est

In [ ]:
# Estimate NTL effects
est_ntl = fit_causal_forest(df, 'ntl_log', 'NTL')

## 5. Average Treatment Effects

EconML's `ate_inference()` returns the average causal effect for a chosen pair of treatment levels with a Bootstrap-of-Little-Bags (BLB) standard error and confidence interval.

The reported SE is the SE of the *forest-level* ATE point estimate, not the SE of any one unit's CATE. BLB exploits tree-level conditional independence to estimate this SE without refitting hundreds of full forests; it is enabled whenever `inference=True` is passed to the constructor. We report 90% intervals (`alpha=0.1`) — the convention in Athey, Tibshirani & Wager (2019) and Hodler, Lechner & Raschky (2023).

In [ ]:
X = df[X_COLS].values
gt = expected_ates_ntl()

comparisons = [
    ('1-0', 0, 1), ('2-0', 0, 2), ('3-0', 0, 3),
    ('2-1', 1, 2), ('3-1', 1, 3), ('3-2', 2, 3),
]

print(f"{'Comp':<8s} {'NTL Effect':>11s} {'NTL SE':>8s} "
      f"{'90% CI':>22s} {'Ground Truth':>13s} {'Sig':>5s}")
print('-' * 70)

for comp_label, t0, t1 in comparisons:
    res = est_ntl.ate_inference(X, T0=t0, T1=t1)
    lo, hi = res.conf_int_mean(alpha=0.1)  # 90% CI
    sig = ''
    if res.stderr_mean > 0:
        z = abs(res.mean_point / res.stderr_mean)
        if z > 2.576: sig = '***'
        elif z > 1.96: sig = '**'
        elif z > 1.645: sig = '*'
    truth = gt.get(comp_label, float('nan'))
    ci = f'[{lo:.3f}, {hi:.3f}]'
    print(f'{comp_label:<8s} {res.mean_point:>11.4f} {res.stderr_mean:>8.4f} '
          f'{ci:>22s} {truth:>13.3f} {sig:>5s}')

print('\n* p<0.10, ** p<0.05, *** p<0.01 (two-sided, BLB SEs)')

## 6. Treatment Effect Heterogeneity: GATEs

EconML returns per-observation CATEs through `effect_inference()`. To form a **Group Average Treatment Effect** for a subgroup $g$ we (1) average the CATEs within $g$ and (2) propagate the per-observation BLB standard errors:

$$\widehat{\mathrm{GATE}}_g = \frac{1}{n_g} \sum_{i \in g} \widehat\tau(\mathbf{X}_i),$$

$$\mathrm{Var}(\widehat{\mathrm{GATE}}_g) \approx \frac{1}{n_g^2} \sum_{i \in g} \mathrm{SE}_i^2 = \frac{1}{n_g} \cdot \overline{\mathrm{SE}_i^2},$$

so the GATE standard error is `sqrt(mean(se_i**2) / n_g)`. The within-group independence assumption probably understates the SE on panel data; the qualitative pattern is robust to that caveat.

In [ ]:
def compute_gate(est, df, z_var, t0, t1):
    """Group Average Treatment Effects with BLB SE propagation.

    Mirrors script.py:compute_gate. Update both together.

    Treats per-observation CATE estimates as approximately uncorrelated
    within group (a working assumption — EconML's BLB does not return
    their full covariance matrix), so
        Var(GATE_g) ~= (1/n_g) * mean_{i in g}(se_i**2),
    and SE(GATE_g) = sqrt(mean(se_i**2) / n_g).
    """
    X = df[X_COLS].values
    inf = est.effect_inference(X, T0=t0, T1=t1)
    ite = inf.point_estimate
    ite_se = inf.stderr

    gate_data = []
    for z in sorted(df[z_var].unique()):
        mask = df[z_var].values == z
        n = mask.sum()
        gate_mean = ite[mask].mean()
        gate_se = np.sqrt(np.mean(ite_se[mask] ** 2) / n)
        gate_data.append({
            'z_value': z,
            'gate': gate_mean,
            'se': gate_se,
            'lower': gate_mean - 1.645 * gate_se,
            'upper': gate_mean + 1.645 * gate_se,
            'n': n,
        })
    return pd.DataFrame(gate_data), ite

### GATEs by Executive Constraints (cf. Paper Figure 2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.patch.set_linewidth(0)

panels = [
    (est_ntl, 0, 1, 'NTL: Mining vs No Mining (1-0)', axes[0], STEEL_BLUE),
    (est_ntl, 1, 3, 'NTL: High vs Low Prices (3-1)', axes[1], STEEL_BLUE),
]

for est, t0, t1, title, ax, color in panels:
    gate_df, ite = compute_gate(est, df, 'exec_constraints', t0, t1)
    ax.fill_between(gate_df['z_value'], gate_df['lower'], gate_df['upper'],
                    alpha=0.25, color=color)
    ax.plot(gate_df['z_value'], gate_df['gate'], 'o-',
            color=WHITE_TEXT, markersize=7, linewidth=1.5,
            markeredgecolor=DARK_NAVY, markeredgewidth=0.8, zorder=3)
    ate_val = ite.mean()
    ax.axhline(ate_val, color=WARM_ORANGE, linewidth=1.5, linestyle='--',
               alpha=0.8, label=f'ATE = {ate_val:.3f}')
    ax.set_xlabel('Constraints on the Executive')
    ax.set_ylabel('GATE')
    ax.set_title(title)
    ax.legend(fontsize=10)

plt.suptitle('GATEs by Constraints on the Executive (EconML CausalForestDML)',
             fontsize=14, fontweight='bold', color=WHITE_TEXT, y=1.02)
plt.tight_layout()
plt.show()

### GATEs by Quality of Government (cf. Paper Figure 3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.patch.set_linewidth(0)

panels = [
    (est_ntl, 0, 1, 'NTL: Mining vs No Mining (1-0)', axes[0], STEEL_BLUE),
    (est_ntl, 1, 3, 'NTL: High vs Low Prices (3-1)', axes[1], STEEL_BLUE),
]

for est, t0, t1, title, ax, color in panels:
    gate_df, ite = compute_gate(est, df, 'quality_of_govt', t0, t1)
    ax.fill_between(gate_df['z_value'], gate_df['lower'], gate_df['upper'],
                    alpha=0.25, color=color)
    ax.plot(gate_df['z_value'], gate_df['gate'], 'o-',
            color=WHITE_TEXT, markersize=7, linewidth=1.5,
            markeredgecolor=DARK_NAVY, markeredgewidth=0.8, zorder=3)
    ate_val = ite.mean()
    ax.axhline(ate_val, color=WARM_ORANGE, linewidth=1.5, linestyle='--',
               alpha=0.8, label=f'ATE = {ate_val:.3f}')
    ax.set_xlabel('Quality of Government')
    ax.set_ylabel('GATE')
    ax.set_title(title)
    ax.legend(fontsize=10)

plt.suptitle('GATEs by Quality of Government (EconML CausalForestDML)',
             fontsize=14, fontweight='bold', color=WHITE_TEXT, y=1.02)
plt.tight_layout()
plt.show()

### Finding 3: Institutions moderate the mining margin, not the price margin

**Mining-effect GATEs (1-0)** climb roughly monotonically from $\sim 0.18$ at the weakest institutions to $\sim 0.26$ at the strongest — a range of about 0.09. Weaker institutions cut the development gain from mining roughly in half.

**Price-effect GATEs (3-1)** span only about 0.05 and show no monotone pattern. The forest's flat line on the right panel is itself the finding: by construction in the DGP, the price step is uniform across institutional environments. The asymmetry — institutions shaping the mining margin but not the price margin — is the structural prediction of the institutions-and-resources literature (Mehlum, Moene & Torvik, 2006).

In [ ]:
# GATE values table
print("GATE values for NTL by Executive Constraints:")
print("=" * 65)

for (t0, t1), comp_label in [((0, 1), 'Mining vs No Mining (1-0)'),
                               ((1, 3), 'High vs Low Prices (3-1)')]:
    gate_df, _ = compute_gate(est_ntl, df, 'exec_constraints', t0, t1)
    print(f"\n  {t1}-{t0} ({comp_label}):")
    print(f"  {'Exec. Constr.':<15s} {'GATE':>8s} {'90% CI':>20s} {'N':>6s}")
    print(f"  {'-'*52}")
    for _, row in gate_df.iterrows():
        print(f"  {row['z_value']:>13.0f}   {row['gate']:>8.3f}   "
              f"[{row['lower']:.3f}, {row['upper']:.3f}] {row['n']:>6.0f}")
    rng = gate_df['gate'].max() - gate_df['gate'].min()
    print(f"  Range: {rng:.3f}")

print("\n  1-0: effects INCREASE with institutions (range > 0)")
print("  3-1: effects are FLAT across institutions (range ~ 0)")

## 7. Variable Importance vs Moderation

EconML's `feature_importances_` reports the variance-reduction-weighted frequency with which each $X$-variable is used as a split. **It is not the same as moderation.** A variable $X_j$ is a **moderator** if

$$\frac{\partial \tau(\mathbf{x})}{\partial x_j} \neq 0.$$

Continuous variables (e.g., distance, ethnic fractionalization) accumulate importance because they offer many candidate split thresholds, even when each split contributes little to actual heterogeneity. Coarse discrete variables like `exec_constraints` (6 levels) have at most 5 splits and rank low even when they are the dominant moderator. Read importances as a *screening* signal; confirm moderation with GATEs.

In [ ]:
importances = est_ntl.feature_importances_
vim_data = sorted(zip(X_COLS, importances), key=lambda x: x[1], reverse=True)

print("Feature importances (heterogeneity drivers):")
for var, imp in vim_data:
    bar = '#' * int(imp * 100)
    print(f"  {var:<25s} {imp:>6.3f}  {bar}")

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_linewidth(0)
vars_, imps = zip(*reversed(vim_data))
ax.barh(vars_, imps, color=STEEL_BLUE, edgecolor=DARK_NAVY,
        linewidth=0.8, alpha=0.9)
ax.set_xlabel('Feature Importance (heterogeneity contribution)')
ax.set_title('Treatment Effect Heterogeneity Drivers (NTL)',
             fontweight='bold')
plt.tight_layout()
plt.show()

print("\nNote: These measure which features drive treatment effect")
print("HETEROGENEITY, not outcome prediction.")

## 8. CATE Interpreter

`SingleTreeCateInterpreter` fits a *shallow* decision tree to the estimated CATEs themselves — the tree's outcome is $\widehat\tau(\mathbf{X}_i)$, not $Y_i$. By splitting on $\mathbf{X}$, the tree finds the covariates and thresholds that best separate units with different treatment effects.

Two design choices: **tree depth** trades detail against communicability (depth 2 → 4 leaves, narratable; depth 4+ is diagnostic), and **min_samples_leaf** prevents the tree from carving out tiny noisy subgroups. The CATE Interpreter is the *exploratory* dual of GATEs: GATEs test pre-specified moderators; the tree audits the data for unexpected ones.

In [ ]:
from econml.cate_interpreter import SingleTreeCateInterpreter

CATE_TREE_DEPTH = 2
CATE_TREE_MIN_LEAF = 100

intrp = SingleTreeCateInterpreter(
    max_depth=CATE_TREE_DEPTH,
    min_samples_leaf=CATE_TREE_MIN_LEAF,
)
intrp.interpret(est_ntl, df[X_COLS].values)

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_linewidth(0)
ax.set_facecolor('#f8f9fa')
intrp.plot(feature_names=X_COLS, ax=ax)
plt.title('Interpretable CATE Subgroups: Mining vs No Mining (1-0)',
          fontweight='bold', color=NEAR_BLACK)
plt.tight_layout()
plt.show()

## 9. Connection to the Paper

| Finding | Paper (Table 2, M=4) | Tutorial (DGP) | Pattern Match? |
|---------|---------------------|----------------|:-:|
| ATE(1-0) NTL | 0.195*** | 0.250 | Direction |
| ATE(2-1) NTL | 0.053 (n.s.) | 0.050 | Small, insignificant |
| ATE(3-1) NTL | 0.278*** | 0.300 | Large, significant |
| GATE(1-0) varies with inst. | Yes (Figs 2-3) | Yes (by design) | Upward slope |
| GATE(3-1) flat in inst. | Yes (Figs 2-3) | Yes (by design) | Flat |

### EconML vs MCF Comparison

| Feature | EconML | MCF |
|---------|:------:|:---:|
| Estimator | CausalForestDML | ModifiedCausalForest |
| Framework | DML (residualize) | Direct forest |
| Nuisance models | GBM (explicit) | Internal |
| Inference | BLB | Bootstrap |
| GATE computation | Manual | Built-in |
| Clustered SEs | No (GroupKFold) | Yes (built-in) |
| Feature importance | Built-in attr | OOB %-lost |
| CATE interpreter | Yes | No |

## 10. Conclusion

### Three Takeaways

1. **EconML's CausalForestDML is a powerful tool for heterogeneity analysis.** The DML framework provides Neyman-orthogonal estimation (robust to first-stage errors), and the causal forest captures non-linear heterogeneous effects.

2. **Institutions matter for mining, not for prices.** Stronger institutions increase the economic benefits and reduce the conflict costs of mining activities. But they do not shape the effects of mineral price changes.

3. **Different causal ML frameworks yield similar conclusions.** Both the MCF and EconML recover the same qualitative patterns from the data, providing robustness evidence.

### Exercises

1. Compare nuisance models: Replace GBM with `RandomForestRegressor`. How do results change?
2. Vary trees: Try `n_estimators=100` vs `n_estimators=1000`.
3. Remove `groups=df['district_id'].values` from `fit()`. What happens to the standard errors? Why?
4. Modify the GATE analysis: Use `quality_of_govt` quartiles instead of raw values. Do the patterns become clearer?
5. Add a new outcome: Create `y_new = ntl_log + 0.1 * conflict` and estimate effects on it.

In [ ]:
# Session info
import sys
print(f"Python: {sys.version}")
print(f"econml: {econml.__version__}")
import sklearn; print(f"scikit-learn: {sklearn.__version__}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
import matplotlib; print(f"matplotlib: {matplotlib.__version__}")